# Genotype–Phenotype Heterogeneity Dataset Exploration with `mlcroissant`

This notebook guides you through loading and exploring the FAIR²-structured clinical genotype–phenotype dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access the metadata object
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"Identifier: {metadata.identifier}")
print(f"Keywords: {metadata.keywords}")
print(f"Date Published: {metadata.datePublished}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We'll enumerate the record sets defined by their `@id`, along with their fields (and columns), also referenced by `@id`.

**Note:** In Croissant, record sets define tables or collections, fields are mapped to columns. All references use their `@id`.

In [ ]:
# Display overview of record sets, fields, and columns by @id
record_sets = []
for rs in dataset.record_sets():
    print(f"RecordSet @id: {rs['@id']} | name: {rs.get('name', 'N/A')}")
    record_sets.append(rs['@id'])
    if 'field' in rs:
        for field in rs['field']:
            print(f"  Field @id: {field['@id']} | name: {field.get('name', 'N/A')} | dataType: {field.get('dataType', 'N/A')}")
            if 'column' in field:
                for col in field['column']:
                    print(f"    Column @id: {col['@id']} | name: {col.get('name', 'N/A')}")
    print('-'*40)

## 3. Data Extraction
Load data from each record set into a pandas DataFrame for analysis.

We will use the identified `@id`s from the overview, referencing each record set by its `@id`.

In [ ]:
# Extract data from all record sets
dataframes = {}
for record_set_id in record_sets:
    print(f"Loading records for record set @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Columns: {df.columns.tolist()}")
    print(df.head())
    print('-'*40)

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering, normalizing, grouping records. All fields are referenced by their `@id`.

### Example: Filtering, Normalizing, and Grouping
Select a numeric field (such as hormone level or age) for this demonstration.

In [ ]:
# Example: Assume first record set contains age and hormone levels

# For demonstration, select first available record set and numeric fields
record_set_id = record_sets[0] if record_sets else None
df = dataframes.get(record_set_id, pd.DataFrame())
print(f"Working on record set: {record_set_id}")

# Try to detect 'age' or 'hormone' column by @id or name:
numeric_field = None
group_field = None

# Attempt to find candidate numeric and group fields
for col in df.columns:
    if 'age' in col.lower():
        numeric_field = col
    elif 'hormone' in col.lower():
        numeric_field = col
    if 'phenotype' in col.lower() or 'diagnosis' in col.lower():
        group_field = col
    if 'infertility' in col.lower():
        group_field = col

# If not found, default to first numeric column
if numeric_field is None:
    numeric_types = df.select_dtypes(include='number').columns.tolist()
    numeric_field = numeric_types[0] if numeric_types else None

if numeric_field is not None:
    print(f"Using numeric field: {numeric_field}")
    # Set a threshold for demonstration
    threshold = 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by group_field (e.g., phenotype/infertility)
    if group_field and group_field in df.columns:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by {group_field}:")
        print(grouped_df.head())
else:
    print("No numeric field detected for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields. We'll plot distributions for the selected numeric field and examine relationships with a grouping variable (when present).

If no numeric field is available, skip plotting.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field and not df.empty:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field].dropna(), bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If group field available, visualize group comparisons
    if group_field and group_field in df.columns:
        plt.figure(figsize=(7,4))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
In this notebook, we explored the FAIR² clinical genotype–phenotype dataset using the `mlcroissant` library, referencing all entities by their `@id`. We loaded the Croissant metadata, listed record sets, and extracted tabular data. Initial EDA and visualization demonstrated key features such as filtering by numeric fields (e.g., age at diagnosis or hormone levels), normalization, and grouping by clinical attributes like phenotype or infertility status.

This approach enables reproducible and FAIR-compliant analysis of biomedical datasets. For further research, deeper domain-specific processing, larger cohorts, and integration with broader clinical datasets are recommended.